In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="パッケージバージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
オプションを使用してEstimatorプリミティブをカスタマイズできます。プリミティブの`run()`メソッドのインターフェースはすべての実装で共通ですが、オプションは共通ではありません。[`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2)と[`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html)のオプションについては、APIリファレンスを参照してください。

<span id="pass-options"></span>

## Estimatorオプションの設定

Estimatorの初期化時、初期化後、またはEstimatorの初期化後にオプションを更新することができます。これらの手法の使い方については、[オプションの概要](/guides/runtime-options-overview#options-precedence)トピックを参照してください。

また、次のセクションで説明するように、`run()`メソッドで`precision`値を設定することもできます。
<span id="run-method"></span>

### run()メソッド

`run()`に渡すことができる値は、インターフェースで定義されたもののみです。つまり、EstimatorのPromise`precision`です。これにより、現在の実行で`default_precision`に設定された値が上書きされます。

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### 特殊ケース：precision
`EstimatorV2.run`メソッドは2つの引数を受け付けます：PUBのリスト（各PUBはprecisionのPUB固有の値を指定できます）とprecisionキーワード引数です。これらのprecision値はEstimator実行インターフェースの一部であり、Runtime Estimatorのオプションとは独立しています。Estimatorの抽象化に準拠するために、オプションとして指定された値よりも優先されます。

ただし、PUBまたはrunキーワード引数でprecisionが指定されていない場合（またはすべてが`None`の場合）、オプションのprecision値が使用されます。最も注目すべきは`default_precision`です。

Estimatorオプションには`default_shots`と`default_precision`の両方が含まれていることに注意してください。ただし、gate-twirlingはデフォルトで有効になっているため、`num_randomizations`と`shots_per_randomization`の積がこの2つのオプションよりも優先されます。

具体的には、EstimatorのPUBに対して：

1. PUBがprecisionを指定している場合、その値を使用する。
2. `run`のprecisionキーワード引数が指定されている場合、その値を使用する。
3. `twirling`が有効な場合（デフォルトでTrue）、[`twirling`オプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)で指定された`num_randomizations`と`shots_per_randomization`の積を使用する。
4. `estimator.options.default_shots`が指定されている場合、その値を使用してデータ量を制御する。
5. `estimator.options.default_precision`が指定されている場合、その値を使用する。

たとえば、4つの場所すべてでprecisionが指定されている場合、最高の優先度を持つもの（PUBで指定されたprecision）が使用されます。

> **Note:** 精度は使用量と逆の関係があります。つまり、精度が低いほど、実行に多くのQPU時間がかかります。

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## すべてのエラー軽減とエラー抑制をオフにする
たとえば、独自の軽減技術を研究している場合など、すべてのエラー軽減と抑制をオフにすることができます。これを実現するには、`resilience_level = 0`を設定します。

例：

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## 利用可能なオプション
次の表は、`qiskit-ibm-runtime`の最新バージョンのオプションを文書化しています。古いオプションバージョンを確認するには、[`qiskit-ibm-runtime` APIリファレンス](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime)にアクセスして以前のバージョンを選択してください。

<Accordion>
<AccordionItem title="`default_shots`">

circuit設定ごとに使用する総shot数。

**選択肢**: 整数 >= 0

**デフォルト**: None

[`default_shots` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

精度を指定しないPUBまたは`run()`呼び出しに使用するデフォルトの精度。

**選択肢**: 浮動小数点数 > 0

**デフォルト**: 0.015625 (1 / sqrt(4096))

[`default_precision` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

dynamical decouplingエラー軽減設定を制御します。

[`dynamical_decoupling` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**選択肢**: `True`, `False`

**デフォルト**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**選択肢**: `middle`, `edges`

**デフォルト**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

選択肢: `asap`, `alap`
デフォルト: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

選択肢: `XX`, `XpXm`, `XY4`
デフォルト: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

選択肢: `True`, `False`
デフォルト: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[`environment` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

`Job ID`と`Job result`を受け取るコール可能関数。

**選択肢**: None

**デフォルト**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

タグのリスト。

**選択肢**: None

**デフォルト**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**選択肢**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**デフォルト**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**選択肢**: `True`, `False`

**デフォルト**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[`execution` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
各shotに対してqubitを基底状態にリセットするかどうか。

**選択肢**: `True`, `False`

**デフォルト**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
測定とその後の量子circuitの間の遅延。

**選択肢**: `backend.rep_delay_range`によって提供される範囲の値

**デフォルト**: `backend.default_rep_delay`で指定
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
ジョブの実行時間の制限（秒単位）。詳細については、[最大実行時間](/guides/max-execution-time)ガイドを参照してください。

**選択肢**: [1, 10800]の範囲の整数（秒）

**デフォルト**: 10800（3時間）
  </AccordionItem>

<AccordionItem title="`resilience`">
resilience戦略を細かく調整するための高度なresilienceオプション。

[`resilience` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

レイヤーノイズ学習のオプション。

[`resilience.layer_noise_learning` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**選択肢**: [0, 200]の範囲の2〜10個の値のリスト[int]

**デフォルト**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**選択肢**: None, 整数 >= 1

**デフォルト**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**選択肢**: 整数 >= 1

**デフォルト**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**選択肢**: 整数 >= 1

**デフォルト**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**選択肢**: `NoiseLearnerResult`, `Sequence[LayerError]`

**デフォルト**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**選択肢**: `True`, `False`

**デフォルト**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

測定ノイズ学習のオプション。

[`resilience.measure_noise_learning` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**選択肢**: 整数 >= 1

**デフォルト**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**選択肢**: 整数, `auto`

**デフォルト**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**選択肢**: `True`, `False`

**デフォルト**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

確率的エラーキャンセル軽減オプション。

[`resilience.pec` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**選択肢**: `None`, 整数 >= 1

**デフォルト**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**選択肢**: `auto`, [0, 1]の範囲の浮動小数点数

**デフォルト**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**選択肢**: `True`, `False`

**デフォルト**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[`resilience.zne` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**選択肢**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**デフォルト**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**選択肢**: 浮動小数点数のリスト

**デフォルト**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**選択肢**: 1つ以上: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**デフォルト**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**選択肢**: 浮動小数点数のリスト; 各浮動小数点数 >= 1

**デフォルト**: `PEA`の場合`(1, 1.5, 2)`、それ以外は`(1, 3, 5)`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

エラーに対してどれだけのresilienceを構築するか。レベルが高いほど、より長い処理時間と引き換えに、より正確な結果が得られます。詳細については、Noise managementトピックの[resilienceレベル](/guides/estimator-noise-management#resilience)セクションを参照してください。

**選択肢**: `0`, `1`, `2`

**デフォルト**: `1`

[`resilience_level` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**選択肢**: 整数

**デフォルト**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

バックエンドをシミュレートする際に渡すオプション

[`simulator` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**選択肢**: アンロール先の基底ゲート名のリスト

**デフォルト**: [Qiskit Aerシミュレータ](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)がサポートするすべての基底ゲートのセット

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**選択肢**: 有向の2-qubit相互作用のリスト

**デフォルト**: None（接続制約なし、完全接続を意味します）

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**選択肢**: [Qiskit Aer NoiseModel](/guides/build-noise-models)、またはその表現

**デフォルト**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**選択肢**: 整数

**デフォルト**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Twirlingオプション

[`twirling` APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**選択肢**: True, False

**デフォルト**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**選択肢**: True, False

**デフォルト**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**選択肢**: `auto`, 整数 >= 1

**デフォルト**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**選択肢**: `auto`, 整数 >= 1

**デフォルト**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**選択肢**: `active`, `active-circuit`, `active-accum`, `all`

**デフォルト**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

実験的オプション（利用可能な場合）。

  </AccordionItem>

</Accordion>
## 機能の互換性
特定のランタイム機能は、単一のジョブ内では一緒に使用できません。互換性のない機能のリストを確認するには、適切なタブをクリックしてください：

<Tabs>

  <TabItem value="Fractional" label="分数ゲート">
  次のものとは互換性がありません：
  - ゲートtwirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="ゲートフォールディングZNE">
    次のものとは互換性がありません：
  - PEA
  - PEC

  カスタムゲートを使用すると動作しない場合があります。
  </TabItem>
  <TabItem value="Twirling" label="ゲートtwirling">
  分数ゲートまたはストレッチとは互換性がありません。

  その他の注意事項：
  - 測定twirlingは端末測定にのみ適用できます。
  - 非Cliffordエンタングラーでは動作しません。

  </TabItem>

  <TabItem value="PEA" label="PEA">
    次のものとは互換性がありません：
  - 分数ゲート
  - ゲートフォールディングZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    次のものとは互換性がありません：
  - 分数ゲート
  - ゲートフォールディングZNE
  - PEA
  </TabItem>

</Tabs>
## 次のステップ
> **Tip:** - [オプションの概要](/guides/runtime-options-overview)ガイドを確認する。
> - `EstimatorV2`メソッドの詳細については、[Estimator APIリファレンス](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)を参照する。
> - ジョブを実行する[実行モード](/guides/execution-modes)を決定する。
> - [EstimatorによるNoiseの管理](/guides/estimator-noise-management)について学ぶ。

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>